[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/ACE-Step_Colab.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/ACE-Step_Colab.ipynb)

# 🎸 ACE-Step v1 (3.5B) — Fast Full-Song Music Generation, Audio2Audio Remix, Lyric Editing & Section Repainting (Apache 2.0)

> **Runtime required:** GPU — **T4 (16 GB)**, **L4 (24 GB)**, or **A100 (40/80 GB)**. Synthesizes up to **4 minutes (240 seconds) of 48 kHz stereo music in ~15–20 seconds** on A100 (~45–60s on T4 with `cpu_offload=True` and `overlapped_decode=True`).

A complete Google Colab studio for **[ACE-Step v1 3.5B](https://github.com/ace-step/ACE-Step)** (`ACE-Step/ACE-Step-v1-3.5B`, **Apache 2.0 License**) by **ACE Studio & StepFun** — a fast foundation model for full-song generation combining a **3.5B Diffusion Transformer (`ACEStepTransformer2DModel`)**, **Deep Compression AutoEncoder (`MusicDCAE` + `MusicVocoder` at 48 kHz)**, **UMT5 text encoder**, and **REPA-aligned 19-language `VoiceBpeTokenizer`**.

**Upstream links:** [ace-step/ACE-Step](https://github.com/ace-step/ACE-Step) · [ACE-Step/ACE-Step-v1-3.5B](https://huggingface.co/ACE-Step/ACE-Step-v1-3.5B) · [HF Space](https://huggingface.co/spaces/ACE-Step/ACE-Step) · [Apache 2.0 License](https://github.com/ace-step/ACE-Step/blob/main/LICENSE)

---

## 🎼 Supported Workflows in `ACEStepPipeline`

| Tab / Workflow | Pipeline Task | What It Does |
|---|---|---|
| **1. Text + Lyrics $\rightarrow$ Full Song** | `task="text2music"` | Generates full songs (10–240s) from genre/instrument tags + structured lyrics (`[verse]`, `[chorus]`, `[bridge]`, `[instrumental]`) across 19 languages with APG guidance & preset genre buttons |
| **2. Audio2Audio Remix & Retake** | `audio2audio_enable=True` / `task="retake"` | **Audio2Audio**: Guides new song generation with a reference track (`ref_audio_strength=0.1..0.9`). **Retake**: Creates seed-controlled variations (`retake_variance=0.0..1.0`) |
| **3. Lyric Editing & Section Repainting** | `task="repaint"` / `task="edit"` | **Repaint**: Regenerates only a specific time window (`repaint_start` $\rightarrow$ `repaint_end` seconds). **FlowEdit (`edit`)**: Modifies lyrics or style tags while preserving the original melody & arrangement |

## 💾 Dual Google Drive Output Saving (3D/Video + Avatar/Voice)

- Every generated `.wav` and its companion `_input_params.json` are saved to **both**:
  - `/content/drive/MyDrive/AEI_3D_Out/ACE-Step/` (for muxing with `Wan2.2`, `LTX-2-5`, `MMAudio`)
  - `/content/drive/MyDrive/AEI_TTS_Out/ACE-Step/` (automatically discovered by **`Seed-VC_Colab.ipynb`** for Singing Voice Conversion and **`InfiniteTalk_Colab.ipynb`** / **`LongCat-Video-Avatar-1.5_Colab.ipynb`** for singing avatar videos!)

## 🖥️ GPU & Memory Profile

| GPU | VRAM | Recommended Mode | Peak VRAM | 60s Song Time |
|---|---|---|---|---|
| **T4** | 16 GB | `bfloat16` + `cpu_offload=True` + `overlapped_decode=True` | ~8.5 GB | ~45–65s |
| **L4** | 24 GB | `bfloat16` + `cpu_offload=False` + `overlapped_decode=True` | ~15.5 GB | ~25–35s |
| **A100** | 40 / 80 GB | `bfloat16` + `cpu_offload=False` | ~16.0 GB | ~12–20s |

In [ ]:
# @title STEP 1 — Clone ace-step/ACE-Step & Install Dependencies
# @markdown Mounts Google Drive, configures `HF_HOME` at `/content/drive/MyDrive/AEI_TTS_Cache/ACE-Step`, clones `https://github.com/ace-step/ACE-Step.git`, and installs runtime dependencies (`diffusers>=0.33.0`, `transformers`, `pypinyin`, `cutlet`, `fugashi[unidic-lite]`, `py3langid`, `hangul-romanize`, `gradio>=5.0.0`, `tqdm`).

import os, sys, subprocess, shutil, time
import torch

print("=" * 72)
print("🎸 ACE-Step v1 (3.5B) — Environment Setup & Installation")
print("=" * 72)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[GPU] PyTorch {torch.__version__} · Device = {DEVICE}", flush=True)
if DEVICE == "cuda":
    try:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[GPU] {gpu_name} ({vram_gb:.1f} GB VRAM)", flush=True)
    except Exception as e:
        print(f"[GPU] Could not read GPU info: {e}", flush=True)

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
if MOUNT_GOOGLE_DRIVE:
    try:
        if not os.path.exists("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[Drive] Notice — running with local storage fallback ({e})", flush=True)

if os.path.exists("/content/drive/MyDrive"):
    CACHE_ROOT = "/content/drive/MyDrive/AEI_TTS_Cache/ACE-Step"
    OUT_DIR_3D = "/content/drive/MyDrive/AEI_3D_Out/ACE-Step"
    OUT_DIR_TTS = "/content/drive/MyDrive/AEI_TTS_Out/ACE-Step"
else:
    CACHE_ROOT = "/content/cache/ACE-Step"
    OUT_DIR_3D = "/content/outputs/ACE-Step_3D"
    OUT_DIR_TTS = "/content/outputs/ACE-Step_TTS"

for d in (CACHE_ROOT, OUT_DIR_3D, OUT_DIR_TTS, "/content/acestep_out"):
    os.makedirs(d, exist_ok=True)

os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_ROOT, "hub")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"[Cache] HF_HOME      = {CACHE_ROOT}", flush=True)
print(f"[Out 1] OUT_DIR_3D   = {OUT_DIR_3D}", flush=True)
print(f"[Out 2] OUT_DIR_TTS  = {OUT_DIR_TTS}", flush=True)

if not shutil.which("ffmpeg"):
    print("[apt] Installing ffmpeg...", flush=True)
    subprocess.run(["apt-get", "-qq", "-y", "install", "ffmpeg"], check=False)

REPO_DIR = "/content/ACE-Step"
if not os.path.exists(os.path.join(REPO_DIR, "acestep", "pipeline_ace_step.py")):
    print("\n[git] Cloning https://github.com/ace-step/ACE-Step.git ...", flush=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ace-step/ACE-Step.git", REPO_DIR], check=True)
else:
    print(f"\n[git] Repository already present at {REPO_DIR}", flush=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

t0 = time.time()
print("\n[pip] Installing ACE-Step runtime packages...", flush=True)
PKGS = [
    "diffusers>=0.33.0",
    "transformers>=4.48.0",
    "accelerate>=1.2.0",
    "pypinyin>=0.53.0",
    "pytorch_lightning>=2.4.0",
    "soundfile>=0.12.1",
    "librosa>=0.10.2",
    "loguru>=0.7.2",
    "py3langid>=0.3.0",
    "hangul-romanize>=0.1.0",
    "num2words>=0.5.13",
    "spacy>=3.7.0",
    "cutlet",
    "fugashi[unidic-lite]",
    "click",
    "peft>=0.13.0",
    "huggingface_hub>=0.26.0",
    "tqdm>=4.66.0",
    "gradio>=5.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR], check=False)
print(f"[pip] Installed in {time.time() - t0:.1f}s", flush=True)

print("\n[Verify] Checking ACEStepPipeline import...", flush=True)
try:
    from acestep.pipeline_ace_step import ACEStepPipeline, SUPPORT_LANGUAGES
    print(f"  ✓ ACEStepPipeline imported OK ({len(SUPPORT_LANGUAGES)} lyric languages supported)!")
except Exception as e:
    print(f"  ✗ Import failed: {e}")
    raise

print("\n[Done] STEP 1 complete — proceed to STEP 2.")

In [ ]:
# @title STEP 2 — Pre-Cache ACE-Step-v1-3.5B Checkpoints to Google Drive
# @markdown Downloads `ACE-Step/ACE-Step-v1-3.5B` (`music_dcae_f8c8`, `music_vocoder`, `ace_step_transformer`, `umt5-base`, ~7.8 GB in bf16) into `/content/drive/MyDrive/AEI_TTS_Cache/ACE-Step/checkpoints`.

import os, time
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download

CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/ACE-Step")
os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT

CKPT_DIR = os.path.join(CACHE_ROOT, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

REPO_ID = "ACE-Step/ACE-Step-v1-3.5B"
PRECACHE_CHINESE_RAP_LORA = False  # @param {type:"boolean"}

required_subdirs = ["music_dcae_f8c8", "music_vocoder", "ace_step_transformer", "umt5-base"]
already_ready = all(os.path.isdir(os.path.join(CKPT_DIR, d)) for d in required_subdirs)

if already_ready:
    print(f"[Cache] All 4 ACE-Step component directories already exist in {CKPT_DIR}")
else:
    print(f"[Cache] Downloading {REPO_ID} to {CKPT_DIR} ...")
    t0 = time.time()
    try:
        dl_path = snapshot_download(
            repo_id=REPO_ID,
            cache_dir=CKPT_DIR,
            max_workers=4,
        )
        total_gb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fs in os.walk(dl_path)
            for f in fs
            if os.path.exists(os.path.join(dp, f))
        ) / (1024 ** 3)
        print(f"  ✓ Downloaded {REPO_ID} ({total_gb:.2f} GB) in {time.time() - t0:.1f}s -> {dl_path}")
    except Exception as e:
        print(f"  ⚠ Pre-cache warning ({e}) — ACEStepPipeline will finish downloading on first call.")

if PRECACHE_CHINESE_RAP_LORA:
    try:
        lora_path = snapshot_download("ACE-Step/ACE-Step-v1-chinese-rap-LoRA", cache_dir=CKPT_DIR)
        print(f"  ✓ Cached Chinese Rap LoRA at {lora_path}")
    except Exception as e:
        print(f"  ⚠ Could not cache LoRA: {e}")

print("\n[Done] STEP 2 complete — proceed to STEP 3.")

In [ ]:
# @title STEP 3 — Core ACEStepPipeline Loader & Multi-Task Music Functions
# @markdown Defines the lazy `ACEStepPipeline` singleton and unified `run_acestep()` wrapper supporting:
# @markdown - **`text2music`**: Full-song generation from tags + structured lyrics
# @markdown - **`audio2audio`**: Reference-audio guided remixing (`audio2audio_enable=True`, `ref_audio_strength`)
# @markdown - **`retake`**: Seed/variance variations (`retake_seeds`, `retake_variance`)
# @markdown - **`repaint`**: Time-window audio inpainting (`repaint_start` to `repaint_end`)
# @markdown - **`edit`**: FlowEdit lyric & style editing without altering the rest of the song (`edit_target_prompt`, `edit_target_lyrics`)

import os, sys, gc, time, json, shutil, random, builtins
from pathlib import Path
from typing import Optional, Dict, List
import torch
from tqdm.auto import tqdm

REPO_DIR = "/content/ACE-Step"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from acestep.pipeline_ace_step import ACEStepPipeline

CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/ACE-Step")
os.environ.setdefault("HF_HOME", CACHE_ROOT)
CKPT_DIR = os.path.join(CACHE_ROOT, "checkpoints")

if os.path.exists("/content/drive/MyDrive"):
    OUT_DIR_3D = "/content/drive/MyDrive/AEI_3D_Out/ACE-Step"
    OUT_DIR_TTS = "/content/drive/MyDrive/AEI_TTS_Out/ACE-Step"
else:
    OUT_DIR_3D = "/content/outputs/ACE-Step_3D"
    OUT_DIR_TTS = "/content/outputs/ACE-Step_TTS"

os.makedirs(OUT_DIR_3D, exist_ok=True)
os.makedirs(OUT_DIR_TTS, exist_ok=True)
os.makedirs("/content/acestep_out", exist_ok=True)

CPU_OFFLOAD_MODE = "auto"  # @param ["auto", "true", "false"]
OVERLAPPED_DECODE = True  # @param {type:"boolean"}
TORCH_COMPILE = False  # @param {type:"boolean"}

GENRE_PRESETS = {
    "Synthwave / Retrowave": {
        "prompt": "synthwave, retrowave, 80s analog synthesizer, driving bassline, gated reverb drums, nostalgic, cinematic, 110 bpm, clear female vocal",
        "lyrics": "[intro]\n[instrumental]\n\n[verse]\nNeon reflections on the midnight boulevard\nChasing electric dreams beneath a velvet sky\nEvery heartbeat echoes through the city grid\n\n[chorus]\nWe are running through the starlight tonight\nHolding on to signals in the ultraviolet glow\nNever looking back at what we left behind\n\n[outro]\n[instrumental]",
    },
    "Indie Folk / Acoustic": {
        "prompt": "indie folk, warm acoustic guitar fingerpicking, upright bass, brushed snare, intimate male tenor vocal, uplifting, organic, 96 bpm",
        "lyrics": "[verse]\nMorning light comes pouring through the kitchen pane\nCoffee steam is rising while the sparrows sing\nPages of a notebook waiting for a line\n\n[chorus]\nEvery little road leads back to where we started\nEvery quiet river finds the open sea\nTake a breath and let the morning carry us along",
    },
    "Cyberpunk / Melodic Dubstep": {
        "prompt": "cyberpunk electronic, melodic dubstep, heavy sub bass, supersaw chords, glitch percussion, futuristic, high energy, 140 bpm, powerful female vocal",
        "lyrics": "[intro]\n(arpeggiated synth buildup)\n\n[verse]\nWires hum beneath the chrome horizon\nStatic breaking through the digital rain\n\n[chorus]\nBreak the circuit, light the sky on fire\nRising higher than the towers of steel\n\n[bridge]\n[instrumental]",
    },
    "Lo-Fi Chillhop (Instrumental)": {
        "prompt": "lo-fi hip hop, jazzy Rhodes electric piano, vinyl crackle, mellow boom bap drums, warm sub bass, relaxing study beats, 82 bpm, instrumental",
        "lyrics": "[instrumental]\n[intro]\n(warm Rhodes chords and vinyl crackle)\n[verse]\n(mellow boom bap groove)\n[chorus]\n(soft muted trumpet melody)\n[outro]\n(gentle fade out)",
    },
    "Anime J-Rock / Upbeat Band": {
        "prompt": "anime opening rock, j-rock, fast electric guitar riffs, driving bass, energetic live drums, piano arpeggios, bright expressive female vocal, 175 bpm",
        "lyrics": "[verse]\nRunning past the station as the bell begins to ring\nA thousand paper airplanes flying in the wind\n\n[chorus]\nReach out for the horizon, never let the spark fade out\nEven when the storm clouds gather overhead\nWe will paint tomorrow in our brightest colors!",
    },
    "Cinematic Orchestral Score": {
        "prompt": "epic orchestral soundtrack, sweeping string ensemble, French horns, booming taiko percussion, choir harmonies, heroic, adventure film score, 120 bpm",
        "lyrics": "[instrumental]\n[intro]\n(low strings and distant horn call)\n[verse]\n(building percussion and brass theme)\n[chorus]\n(full orchestra and soaring choir climax)\n[outro]\n(resonant orchestral finale)",
    },
}

_ACE_PIPE: Optional[ACEStepPipeline] = None
_ACE_CFG: dict = {}


def _resolve_cpu_offload() -> bool:
    if CPU_OFFLOAD_MODE == "true":
        return True
    if CPU_OFFLOAD_MODE == "false":
        return False
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        return vram_gb < 20.0
    return False


def get_acestep_pipeline(force_reload: bool = False) -> ACEStepPipeline:
    global _ACE_PIPE, _ACE_CFG
    use_offload = _resolve_cpu_offload()
    cfg = {
        "cpu_offload": use_offload,
        "overlapped_decode": bool(OVERLAPPED_DECODE),
        "torch_compile": bool(TORCH_COMPILE),
    }
    if _ACE_PIPE is not None and not force_reload and _ACE_CFG == cfg:
        return _ACE_PIPE

    if _ACE_PIPE is not None:
        del _ACE_PIPE
        _ACE_PIPE = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"[ACE-Step] Initializing ACEStepPipeline (cpu_offload={use_offload}, overlapped_decode={OVERLAPPED_DECODE}) ...", flush=True)
    t0 = time.time()
    pipe = ACEStepPipeline(
        checkpoint_dir=CKPT_DIR,
        dtype="bfloat16" if torch.cuda.is_available() else "float32",
        torch_compile=bool(TORCH_COMPILE),
        cpu_offload=use_offload,
        overlapped_decode=bool(OVERLAPPED_DECODE),
    )
    _ACE_PIPE = pipe
    _ACE_CFG = cfg
    print(f"[ACE-Step] Pipeline wrapper ready in {time.time() - t0:.1f}s (weights load on first call).", flush=True)
    return pipe


def free_acestep_vram() -> str:
    global _ACE_PIPE
    _ACE_PIPE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, tot_b = torch.cuda.mem_get_info(0)
        return f"ACEStepPipeline unloaded · GPU free: {free_b/1e9:.1f}/{tot_b/1e9:.1f} GB"
    return "ACEStepPipeline unloaded (CPU)"


def discover_drive_songs(max_files: int = 60) -> List[str]:
    roots = [
        OUT_DIR_3D,
        OUT_DIR_TTS,
        "/content/drive/MyDrive/AEI_3D_Out/MiniMax-Music3",
        "/content/drive/MyDrive/AEI_TTS_Out",
    ]
    exts = {".wav", ".mp3", ".flac", ".ogg"}
    found, seen = [], set()
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dp, _, fs in os.walk(r):
            for fn in fs:
                if os.path.splitext(fn)[1].lower() in exts:
                    p = os.path.abspath(os.path.join(dp, fn))
                    if p not in seen:
                        seen.add(p)
                        try:
                            found.append((os.path.getmtime(p), p))
                        except OSError:
                            pass
    found.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in found[:max_files]]


def run_acestep(
    prompt: str,
    lyrics: str = "",
    audio_duration: float = 60.0,
    infer_step: int = 60,
    guidance_scale: float = 15.0,
    scheduler_type: str = "euler",
    cfg_type: str = "apg",
    omega_scale: float = 10.0,
    seed: int = 0,
    task: str = "text2music",
    audio2audio_enable: bool = False,
    ref_audio_input: Optional[str] = None,
    ref_audio_strength: float = 0.5,
    retake_seed: int = 0,
    retake_variance: float = 0.5,
    repaint_start: int = 0,
    repaint_end: int = 30,
    src_audio_path: Optional[str] = None,
    edit_target_prompt: Optional[str] = None,
    edit_target_lyrics: Optional[str] = None,
    edit_n_min: float = 0.0,
    edit_n_max: float = 1.0,
    lora_name_or_path: str = "none",
    lora_weight: float = 1.0,
    prefix: str = "acestep",
) -> Dict[str, object]:
    """Unified wrapper around ACEStepPipeline.__call__ that saves outputs to both
    /content/drive/MyDrive/AEI_3D_Out/ACE-Step/ and /content/drive/MyDrive/AEI_TTS_Out/ACE-Step/."""
    pipe = get_acestep_pipeline()

    actual_seed = int(seed) if (seed is not None and int(seed) > 0) else random.randint(1, 2**31 - 1)
    actual_retake_seed = int(retake_seed) if (retake_seed is not None and int(retake_seed) > 0) else random.randint(1, 2**31 - 1)

    # Normalize task & file arguments so upstream assertions are always satisfied
    ref_audio = ref_audio_input if (ref_audio_input and os.path.exists(str(ref_audio_input))) else None
    src_audio = src_audio_path if (src_audio_path and os.path.exists(str(src_audio_path))) else None

    if audio2audio_enable and ref_audio is not None:
        effective_task = "audio2audio"
        src_audio = None
    elif task == "retake" and src_audio is not None:
        # Upstream pipeline_ace_step.py uses task='repaint' over [0, audio_duration] when a source audio file is given
        effective_task = "repaint"
        repaint_start = 0
        repaint_end = int(audio_duration)
    else:
        effective_task = task

    ts = int(time.time())
    out_wav_3d = os.path.join(OUT_DIR_3D, f"{prefix}_{effective_task}_{int(audio_duration)}s_seed{actual_seed}_{ts}.wav")

    t0 = time.time()
    outputs = pipe(
        format="wav",
        audio_duration=float(audio_duration),
        prompt=prompt or "pop, melodic, stereo 48khz",
        lyrics=lyrics or "",
        infer_step=int(infer_step),
        guidance_scale=float(guidance_scale),
        scheduler_type=str(scheduler_type),
        cfg_type=str(cfg_type),
        omega_scale=float(omega_scale),
        manual_seeds=[actual_seed],
        retake_seeds=[actual_retake_seed],
        retake_variance=float(retake_variance),
        audio2audio_enable=bool(audio2audio_enable and ref_audio is not None),
        ref_audio_strength=float(ref_audio_strength),
        ref_audio_input=ref_audio,
        lora_name_or_path=str(lora_name_or_path or "none"),
        lora_weight=float(lora_weight),
        task=effective_task,
        repaint_start=int(repaint_start),
        repaint_end=int(repaint_end),
        src_audio_path=src_audio,
        edit_target_prompt=edit_target_prompt or prompt,
        edit_target_lyrics=edit_target_lyrics if edit_target_lyrics is not None else lyrics,
        edit_n_min=float(edit_n_min),
        edit_n_max=float(edit_n_max),
        save_path=out_wav_3d,
        batch_size=1,
    )
    elapsed = time.time() - t0

    wav_path_3d = outputs[0] if isinstance(outputs, list) and len(outputs) > 0 else out_wav_3d
    params_json = outputs[-1] if isinstance(outputs, list) and isinstance(outputs[-1], dict) else {}

    # Mirror output WAV and JSON to /content/drive/MyDrive/AEI_TTS_Out/ACE-Step/
    wav_path_tts = os.path.join(OUT_DIR_TTS, os.path.basename(wav_path_3d))
    if os.path.exists(wav_path_3d) and os.path.abspath(wav_path_3d) != os.path.abspath(wav_path_tts):
        try:
            shutil.copy2(wav_path_3d, wav_path_tts)
            json_3d = wav_path_3d.replace(".wav", "_input_params.json")
            if os.path.exists(json_3d):
                shutil.copy2(json_3d, os.path.join(OUT_DIR_TTS, os.path.basename(json_3d)))
        except Exception as ce:
            print(f"[Mirror] Warning copying to AEI_TTS_Out: {ce}")

    return {
        "wav_path": wav_path_3d,
        "wav_path_tts": wav_path_tts,
        "duration_s": float(audio_duration),
        "elapsed_s": elapsed,
        "seed_used": actual_seed,
        "retake_seed_used": actual_retake_seed,
        "task": effective_task,
        "params": params_json,
    }


builtins.run_acestep = run_acestep
builtins.get_acestep_pipeline = get_acestep_pipeline
builtins.free_acestep_vram = free_acestep_vram
builtins.discover_drive_songs = discover_drive_songs
builtins.ACESTEP_GENRE_PRESETS = GENRE_PRESETS
builtins.ACESTEP_OUT_3D = OUT_DIR_3D
builtins.ACESTEP_OUT_TTS = OUT_DIR_TTS

print("[Core] STEP 3 ready. ACEStepPipeline will load on first generation.")
print(f"       Outputs saved to: {OUT_DIR_3D} AND {OUT_DIR_TTS}")

In [ ]:
# @title STEP 4 — Launch Gradio UI (Text2Music, Audio2Audio & Retake, Repaint & FlowEdit)
# @markdown Launches the 3-Tab ACE-Step Music Studio:
# @markdown 1. **Tab 1: Text + Lyrics to Full Song (`text2music`)** — Genre preset buttons, structured lyrics (`[verse]`, `[chorus]`, `[bridge]`, `[instrumental]`), duration up to 240s, guidance/omega controls, LoRA support.
# @markdown 2. **Tab 2: Audio2Audio Remix & Retake (`audio2audio` / `retake`)** — Reference audio remixing (`ref_audio_strength`) or seed-controlled retake variations (`retake_variance`).
# @markdown 3. **Tab 3: Lyric Editing & Section Repainting (`repaint` / `edit`)** — Time-window inpainting (`repaint_start` to `repaint_end`) or FlowEdit lyric replacement without altering the rest of the song.

import os, time
import gradio as gr
from IPython.display import clear_output

init_songs = discover_drive_songs(max_files=50)
SONG_CHOICES = ["(Use uploaded audio above)"] + init_songs


def _resolve_audio(up_path, dr_choice):
    if up_path and os.path.exists(str(up_path)):
        return str(up_path)
    if dr_choice and dr_choice != "(Use uploaded audio above)" and os.path.exists(str(dr_choice)):
        return str(dr_choice)
    return None


def _refresh_songs():
    s = ["(Use uploaded audio above)"] + discover_drive_songs(max_files=50)
    return gr.update(choices=s, value=s[1] if len(s) > 1 else s[0]), gr.update(choices=s, value=s[1] if len(s) > 1 else s[0])


def _ui_text2music(prompt, lyrics, duration, steps, cfg_scale, omega, sched, cfg_type, seed, lora_choice, lora_w, progress=gr.Progress()):
    if not prompt or not prompt.strip():
        raise gr.Error("Enter genre/style tags in the Prompt box.")
    try:
        progress(0.1, desc=f"Synthesizing {int(duration)}s song with ACE-Step...")
        res = run_acestep(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=float(duration),
            infer_step=int(steps),
            guidance_scale=float(cfg_scale),
            omega_scale=float(omega),
            scheduler_type=sched,
            cfg_type=cfg_type,
            seed=int(seed),
            task="text2music",
            lora_name_or_path=lora_choice,
            lora_weight=float(lora_w),
            prefix="acestep_song",
        )
        progress(1.0, desc="Done!")
        info = (
            f"Task: text2music · Duration: {res['duration_s']:.0f}s · Wall time: {res['elapsed_s']:.1f}s ({res['duration_s']/max(res['elapsed_s'], 0.1):.1f}x RT)\n"
            f"Seed: {res['seed_used']} · Steps: {int(steps)} · Scheduler: {sched} ({cfg_type}, CFG={cfg_scale}, omega={omega})\n"
            f"Saved to AEI_3D_Out : {res['wav_path']}\n"
            f"Saved to AEI_TTS_Out: {res['wav_path_tts']}"
        )
        return res["wav_path"], res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"ACE-Step text2music failed: {e}")


def _ui_remix_retake(mode, prompt, lyrics, ref_up, ref_dr, ref_strength, retake_var, base_seed, retake_seed, duration, steps, cfg_scale, omega, progress=gr.Progress()):
    audio_path = _resolve_audio(ref_up, ref_dr)
    is_a2a = mode.startswith("Audio2Audio")
    if is_a2a and not audio_path:
        raise gr.Error("Audio2Audio Remix mode requires uploading a reference audio file or selecting one from Google Drive.")
    try:
        progress(0.1, desc=f"Running {mode}...")
        res = run_acestep(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=float(duration),
            infer_step=int(steps),
            guidance_scale=float(cfg_scale),
            omega_scale=float(omega),
            seed=int(base_seed),
            retake_seed=int(retake_seed),
            retake_variance=float(retake_var),
            task="audio2audio" if is_a2a else "retake",
            audio2audio_enable=is_a2a,
            ref_audio_input=audio_path if is_a2a else None,
            ref_audio_strength=float(ref_strength),
            src_audio_path=audio_path if (not is_a2a and audio_path) else None,
            prefix="acestep_a2a" if is_a2a else "acestep_retake",
        )
        progress(1.0, desc="Done!")
        info = (
            f"Task: {res['task']} · Duration: {res['duration_s']:.0f}s · Wall time: {res['elapsed_s']:.1f}s\n"
            f"Base Seed: {res['seed_used']} · Retake Seed: {res['retake_seed_used']} · Variance/Strength: {ref_strength if is_a2a else retake_var}\n"
            f"Saved to AEI_3D_Out : {res['wav_path']}\n"
            f"Saved to AEI_TTS_Out: {res['wav_path_tts']}"
        )
        return res["wav_path"], res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"{mode} failed: {e}")


def _ui_repaint_edit(edit_mode, src_up, src_dr, orig_prompt, orig_lyrics, target_prompt, target_lyrics, rep_start, rep_end, retake_var, edit_n_min, edit_n_max, duration, steps, cfg_scale, seed, progress=gr.Progress()):
    src_path = _resolve_audio(src_up, src_dr)
    if not src_path:
        raise gr.Error("Upload a Source Song or pick a generated song from Google Drive to repaint/edit.")
    is_repaint = edit_mode.startswith("Section Repainting")
    try:
        progress(0.1, desc=f"Running {edit_mode}...")
        res = run_acestep(
            prompt=orig_prompt,
            lyrics=orig_lyrics,
            audio_duration=float(duration),
            infer_step=int(steps),
            guidance_scale=float(cfg_scale),
            seed=int(seed),
            task="repaint" if is_repaint else "edit",
            src_audio_path=src_path,
            repaint_start=int(rep_start),
            repaint_end=int(rep_end),
            retake_variance=float(retake_var),
            edit_target_prompt=target_prompt or orig_prompt,
            edit_target_lyrics=target_lyrics if target_lyrics is not None else orig_lyrics,
            edit_n_min=float(edit_n_min),
            edit_n_max=float(edit_n_max),
            prefix="acestep_repaint" if is_repaint else "acestep_edit",
        )
        progress(1.0, desc="Done!")
        info = (
            f"Task: {res['task']} · Source: {os.path.basename(src_path)}\n"
            f"Window: {int(rep_start)}s–{int(rep_end)}s (repaint) · Edit range: [{edit_n_min:.2f}, {edit_n_max:.2f}]\n"
            f"Saved to AEI_3D_Out : {res['wav_path']}\n"
            f"Saved to AEI_TTS_Out: {res['wav_path_tts']}"
        )
        return res["wav_path"], res["wav_path"], info
    except Exception as e:
        raise gr.Error(f"{edit_mode} failed: {e}")


default_preset = GENRE_PRESETS["Synthwave / Retrowave"]

with gr.Blocks(title="ACE-Step v1 (3.5B) Music Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎸 ACE-Step v1 (3.5B) — Fast Full-Song Music Generation, Remixing, Lyric Editing & Repainting\n"
        "Apache 2.0 · 48 kHz Stereo · Up to 240s Full Songs · Dual Drive Save (`AEI_3D_Out/ACE-Step` + `AEI_TTS_Out/ACE-Step`)"
    )
    banner_md = gr.Markdown()

    with gr.Tabs():
        # ── TAB 1: Text + Lyrics to Full Song (text2music) ──
        with gr.Tab("1. Text + Lyrics to Full Song (text2music)"):
            with gr.Row():
                with gr.Column(scale=3):
                    t1_prompt = gr.Textbox(
                        label="Genre, Instrument, Mood & Vocal Tags (Prompt)",
                        lines=3,
                        value=default_preset["prompt"],
                        info="Comma-separated tags describing genre, instrumentation, tempo (BPM), mood, and vocal timbre.",
                    )
                    t1_lyrics = gr.Textbox(
                        label="Structured Lyrics ([verse], [chorus], [bridge], [instrumental], [intro], [outro])",
                        lines=11,
                        value=default_preset["lyrics"],
                        info="Use structure tags in brackets on their own lines. Use [instrumental] for instrumental tracks.",
                    )
                    gr.Markdown("**Quick Genre Presets (click a button or choose from dropdown to populate tags & lyrics):**")
                    with gr.Row():
                        for _pname, _pdata in GENRE_PRESETS.items():
                            _btn = gr.Button(_pname, size="sm")
                            _btn.click(
                                fn=lambda p=_pdata["prompt"], l=_pdata["lyrics"]: (p, l),
                                inputs=None,
                                outputs=[t1_prompt, t1_lyrics],
                            )
                    t1_preset_dd = gr.Dropdown(
                        choices=list(GENRE_PRESETS.keys()),
                        value="Synthwave / Retrowave",
                        label="Genre Preset Selector",
                        info="Automatically fills the Prompt and Structured Lyrics boxes above.",
                    )
                with gr.Column(scale=2):
                    t1_dur = gr.Slider(10, 240, value=60, step=5, label="Song Duration (seconds)", info="Generate 10s previews up to 240s (4-minute) full songs.")
                    t1_steps = gr.Slider(20, 100, value=60, step=5, label="Inference Steps", info="60 = official default quality; 27–35 = fast preview.")
                    t1_cfg = gr.Slider(1.0, 25.0, value=15.0, step=0.5, label="Guidance Scale (CFG)", info="Default 15.0 for APG guidance.")
                    t1_omega = gr.Slider(1.0, 20.0, value=10.0, step=0.5, label="Omega Scale (Granularity)", info="Controls harmonic/rhythmic detail (default 10.0).")
                    with gr.Accordion("Scheduler, CFG Type & LoRA Adapter", open=False):
                        t1_sched = gr.Dropdown(["euler", "heun", "pingpong"], value="euler", label="Scheduler Type", info="euler is fastest; heun/pingpong offer alternative textures.")
                        t1_cfg_type = gr.Dropdown(["apg", "cfg", "cfg_star"], value="apg", label="CFG Type", info="apg (Adaptive Projected Guidance) is the official default.")
                        t1_seed = gr.Number(value=0, precision=0, label="Seed (0 = random)", info="Set >0 for reproducible generation.")
                        t1_lora = gr.Dropdown(
                            ["none", "ACE-Step/ACE-Step-v1-chinese-rap-LoRA"],
                            value="none",
                            label="LoRA Adapter",
                            info="Optional LoRA weights from Hugging Face.",
                        )
                        t1_lora_w = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="LoRA Weight", info="Adapter strength when a LoRA is selected.")
                    t1_btn = gr.Button("🎵 Generate Full Song", variant="primary")
                    t1_out_audio = gr.Audio(label="Generated Song (48 kHz Stereo)", type="filepath")
                    t1_out_file = gr.File(label="Download WAV")
                    t1_info = gr.Textbox(label="Generation Metadata & Drive Paths", lines=4, interactive=False)

            t1_preset_dd.change(
                fn=lambda k: (GENRE_PRESETS[k]["prompt"], GENRE_PRESETS[k]["lyrics"]),
                inputs=[t1_preset_dd],
                outputs=[t1_prompt, t1_lyrics],
            )
            t1_btn.click(
                _ui_text2music,
                inputs=[t1_prompt, t1_lyrics, t1_dur, t1_steps, t1_cfg, t1_omega, t1_sched, t1_cfg_type, t1_seed, t1_lora, t1_lora_w],
                outputs=[t1_out_audio, t1_out_file, t1_info],
            )

        # ── TAB 2: Audio2Audio Remix & Retake ──
        with gr.Tab("2. Audio2Audio Remix & Retake (audio2audio / retake)"):
            with gr.Row():
                with gr.Column(scale=3):
                    t2_mode = gr.Radio(
                        choices=[
                            "Audio2Audio Remix (guide new generation with reference audio)",
                            "Retake Variation (vary seed / variance to create new takes)",
                        ],
                        value="Audio2Audio Remix (guide new generation with reference audio)",
                        label="Workflow Mode",
                        info="Audio2Audio encodes a reference audio file and blends its structure/vibe; Retake generates variations.",
                    )
                    t2_ref_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Reference Audio (upload .wav/.mp3/.flac)")
                    with gr.Row():
                        t2_ref_dr = gr.Dropdown(
                            choices=SONG_CHOICES,
                            value=SONG_CHOICES[1] if len(SONG_CHOICES) > 1 else SONG_CHOICES[0],
                            label="📂 Or Pick Reference Song from Google Drive (ACE-Step / MiniMax-Music3)",
                            info="Cross-notebook picker for existing songs on Google Drive.",
                        )
                        t2_refresh = gr.Button("🔄 Refresh", scale=0)
                    t2_prompt = gr.Textbox(label="Target Genre / Style Prompt", lines=2, value=default_preset["prompt"], info="Target style tags for the remix or retake.")
                    t2_lyrics = gr.Textbox(label="Lyrics", lines=6, value=default_preset["lyrics"], info="Lyrics for the remix/retake.")
                with gr.Column(scale=2):
                    t2_strength = gr.Slider(0.05, 0.95, value=0.5, step=0.05, label="Reference Audio Strength (Audio2Audio)", info="Higher = closer to reference audio structure; lower = more freedom.")
                    t2_var = gr.Slider(0.05, 1.0, value=0.45, step=0.05, label="Retake Variance (Retake mode)", info="0.2 = subtle variation, 0.5 = balanced new take, 0.8 = major remix.")
                    t2_base_seed = gr.Number(value=42, precision=0, label="Base Seed", info="Original seed to vary.")
                    t2_retake_seed = gr.Number(value=0, precision=0, label="Retake Seed (0 = random)", info="Seed driving the retake noise.")
                    t2_dur = gr.Slider(10, 240, value=60, step=5, label="Duration (seconds)", info="Target output duration.")
                    t2_steps = gr.Slider(20, 100, value=60, step=5, label="Inference Steps", info="Default 60.")
                    t2_cfg = gr.Slider(1.0, 25.0, value=15.0, step=0.5, label="Guidance Scale", info="Default 15.0.")
                    t2_omega = gr.Slider(1.0, 20.0, value=10.0, step=0.5, label="Omega Scale", info="Default 10.0.")
                    t2_btn = gr.Button("🎛️ Run Audio2Audio Remix / Retake", variant="primary")
                    t2_out_audio = gr.Audio(label="Remixed / Retake Output", type="filepath")
                    t2_out_file = gr.File(label="Download WAV")
                    t2_info = gr.Textbox(label="Run Info", lines=4, interactive=False)

            t2_btn.click(
                _ui_remix_retake,
                inputs=[t2_mode, t2_prompt, t2_lyrics, t2_ref_up, t2_ref_dr, t2_strength, t2_var, t2_base_seed, t2_retake_seed, t2_dur, t2_steps, t2_cfg, t2_omega],
                outputs=[t2_out_audio, t2_out_file, t2_info],
            )

        # ── TAB 3: Lyric Editing & Section Repainting (repaint / edit) ──
        with gr.Tab("3. Lyric Editing & Section Repainting (repaint / edit)"):
            with gr.Row():
                with gr.Column(scale=3):
                    t3_mode = gr.Radio(
                        choices=[
                            "Section Repainting (repaint time window [repaint_start .. repaint_end])",
                            "FlowEdit Lyric & Style Editing (change lyrics/tags while keeping melody)",
                        ],
                        value="Section Repainting (repaint time window [repaint_start .. repaint_end])",
                        label="Editing Mode",
                        info="Repaint regenerates a specific time slice; FlowEdit edits lyrics/style across the track.",
                    )
                    t3_src_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Source Song to Edit / Repaint")
                    t3_src_dr = gr.Dropdown(
                        choices=SONG_CHOICES,
                        value=SONG_CHOICES[1] if len(SONG_CHOICES) > 1 else SONG_CHOICES[0],
                        label="📂 Or Pick Source Song from Google Drive",
                        info="Select any previously generated song from ACE-Step.",
                    )
                    with gr.Row():
                        t3_orig_prompt = gr.Textbox(label="Original Prompt", lines=2, value=default_preset["prompt"], info="Original genre/style tags of the source song.")
                        t3_tgt_prompt = gr.Textbox(label="Target Prompt (for FlowEdit)", lines=2, value=default_preset["prompt"], info="Modified style/vocal tags for FlowEdit.")
                    with gr.Row():
                        t3_orig_lyrics = gr.Textbox(label="Original Lyrics", lines=6, value=default_preset["lyrics"], info="Original lyrics of the source track.")
                        t3_tgt_lyrics = gr.Textbox(label="Edited Target Lyrics", lines=6, value=default_preset["lyrics"], info="Updated lyrics to sing in the edited/repainted section.")
                with gr.Column(scale=2):
                    t3_start = gr.Slider(0, 220, value=10, step=1, label="Repaint Start (seconds)", info="Start of the time window to regenerate (repaint mode).")
                    t3_end = gr.Slider(5, 240, value=25, step=1, label="Repaint End (seconds)", info="End of the time window to regenerate (repaint mode).")
                    t3_var = gr.Slider(0.05, 1.0, value=0.5, step=0.05, label="Repaint Variance", info="How much the repainted section diverges from the original.")
                    t3_nmin = gr.Slider(0.0, 0.9, value=0.2, step=0.05, label="FlowEdit n_min", info="Minimum diffusion step ratio for FlowEdit (0.2 keeps melody, 0.0 changes more).")
                    t3_nmax = gr.Slider(0.1, 1.0, value=1.0, step=0.05, label="FlowEdit n_max", info="Maximum diffusion step ratio for FlowEdit (default 1.0).")
                    t3_dur = gr.Slider(10, 240, value=60, step=5, label="Song Duration (seconds)", info="Match the source track's duration.")
                    t3_steps = gr.Slider(20, 100, value=60, step=5, label="Inference Steps", info="Default 60.")
                    t3_cfg = gr.Slider(1.0, 25.0, value=15.0, step=0.5, label="Guidance Scale", info="Default 15.0.")
                    t3_seed = gr.Number(value=0, precision=0, label="Seed (0 = random)", info="Seed for repaint/edit.")
                    t3_btn = gr.Button("✂️ Run Section Repaint / Lyric Edit", variant="primary")
                    t3_out_audio = gr.Audio(label="Repainted / Edited Song", type="filepath")
                    t3_out_file = gr.File(label="Download WAV")
                    t3_info = gr.Textbox(label="Edit Info", lines=4, interactive=False)

            t2_refresh.click(_refresh_songs, outputs=[t2_ref_dr, t3_src_dr])
            t3_btn.click(
                _ui_repaint_edit,
                inputs=[t3_mode, t3_src_up, t3_src_dr, t3_orig_prompt, t3_orig_lyrics, t3_tgt_prompt, t3_tgt_lyrics, t3_start, t3_end, t3_var, t3_nmin, t3_nmax, t3_dur, t3_steps, t3_cfg, t3_seed],
                outputs=[t3_out_audio, t3_out_file, t3_info],
            )

    def _on_load():
        return (
            f"✅ **ACE-Step v1 (3.5B) Ready** · All songs are saved to both `{OUT_DIR_3D}` and `{OUT_DIR_TTS}` "
            f"(ready for `Seed-VC` singing conversion and `InfiniteTalk` music videos!)."
        )

    demo.load(_on_load, inputs=None, outputs=[banner_md])

clear_output()
demo.queue(concurrency_limit=1).launch(share=True, inline=False, show_error=True)

In [ ]:
# @title STEP 5 — Keep Colab Session Alive
# @markdown Runs a lightweight background heartbeat thread to prevent Google Colab from disconnecting during interactive sessions.

import threading, time

def _acestep_keepalive():
    while True:
        time.sleep(60)

threading.Thread(target=_acestep_keepalive, daemon=True).start()
print("[Keep-Alive] Heartbeat active. Dual Drive outputs:", OUT_DIR_3D, "|", OUT_DIR_TTS)

In [ ]:
# @title STEP 6 — Quick Test (Single Full-Song Generation without UI)
# @markdown Generates a song from form parameters using `ACEStepPipeline`, plays it inline, and provides a `FileLink` download.

GENRE_PRESET = "Synthwave / Retrowave"  # @param ["Synthwave / Retrowave", "Indie Folk / Acoustic", "Cyberpunk / Melodic Dubstep", "Lo-Fi Chillhop (Instrumental)", "Anime J-Rock / Upbeat Band", "Cinematic Orchestral Score", "Custom"]
CUSTOM_PROMPT = ""  # @param {type:"string"}
CUSTOM_LYRICS = ""  # @param {type:"string"}
DURATION_SECONDS = 45  # @param {type:"slider", min:15, max:240, step:5}
INFER_STEPS = 50  # @param {type:"slider", min:20, max:100, step:5}
GUIDANCE_SCALE = 15.0  # @param {type:"slider", min:1.0, max:25.0, step:0.5}
OMEGA_SCALE = 10.0  # @param {type:"slider", min:1.0, max:20.0, step:0.5}
SCHEDULER_TYPE = "euler"  # @param ["euler", "heun", "pingpong"]
CFG_TYPE = "apg"  # @param ["apg", "cfg", "cfg_star"]
SEED = 42  # @param {type:"integer"}

import os
from IPython.display import Audio, FileLink, display

if GENRE_PRESET != "Custom" and GENRE_PRESET in GENRE_PRESETS:
    p_text = CUSTOM_PROMPT.strip() or GENRE_PRESETS[GENRE_PRESET]["prompt"]
    l_text = CUSTOM_LYRICS.strip() or GENRE_PRESETS[GENRE_PRESET]["lyrics"]
else:
    p_text = CUSTOM_PROMPT.strip() or GENRE_PRESETS["Synthwave / Retrowave"]["prompt"]
    l_text = CUSTOM_LYRICS.strip() or GENRE_PRESETS["Synthwave / Retrowave"]["lyrics"]

print(f"[QuickTest] Generating {DURATION_SECONDS}s song ({GENRE_PRESET}, steps={INFER_STEPS}, seed={SEED}) ...")
res = run_acestep(
    prompt=p_text,
    lyrics=l_text,
    audio_duration=DURATION_SECONDS,
    infer_step=INFER_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    omega_scale=OMEGA_SCALE,
    scheduler_type=SCHEDULER_TYPE,
    cfg_type=CFG_TYPE,
    seed=SEED,
    task="text2music",
    prefix="quicktest_song",
)

print(f"✓ Saved (AEI_3D_Out) : {res['wav_path']}")
print(f"✓ Saved (AEI_TTS_Out): {res['wav_path_tts']} (wall={res['elapsed_s']:.1f}s)")
display(Audio(res["wav_path"], autoplay=False))
display(FileLink(res["wav_path"]))

In [ ]:
# @title STEP 7 — Batch Album / Playlist Generation with tqdm
# @markdown Generates multiple songs from a JSON list (or default multi-genre starter playlist) with per-song `try`/`except` error handling and `tqdm` progress bar.

BATCH_JSON_PATH = "/content/drive/MyDrive/AEI_TTS_Cache/ACE-Step/batch_playlist.json"  # @param {type:"string"}
DEFAULT_DURATION = 45  # @param {type:"slider", min:15, max:240, step:5}
DEFAULT_STEPS = 50  # @param {type:"slider", min:20, max:100, step:5}
BASE_SEED = 1000  # @param {type:"integer"}

import os, json, time
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import Audio, FileLink, display

b_path = Path(BATCH_JSON_PATH)
if not b_path.exists():
    b_path.parent.mkdir(parents=True, exist_ok=True)
    starter_playlist = [
        {
            "title": "01_neon_boulevard",
            "prompt": GENRE_PRESETS["Synthwave / Retrowave"]["prompt"],
            "lyrics": GENRE_PRESETS["Synthwave / Retrowave"]["lyrics"],
            "duration": DEFAULT_DURATION,
            "steps": DEFAULT_STEPS,
        },
        {
            "title": "02_rainy_study_beats",
            "prompt": GENRE_PRESETS["Lo-Fi Chillhop (Instrumental)"]["prompt"],
            "lyrics": GENRE_PRESETS["Lo-Fi Chillhop (Instrumental)"]["lyrics"],
            "duration": DEFAULT_DURATION,
            "steps": DEFAULT_STEPS,
        },
    ]
    b_path.write_text(json.dumps(starter_playlist, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[Batch] Created starter playlist JSON at {b_path}")

items = json.loads(b_path.read_text(encoding="utf-8"))
print(f"[Batch] Generating {len(items)} track(s) ...")

generated_files = []
for idx, item in enumerate(tqdm(items, desc="Step 7 ACE-Step Batch")):
    title = item.get("title", f"track_{idx+1:02d}")
    prompt = item.get("prompt", GENRE_PRESETS["Synthwave / Retrowave"]["prompt"])
    lyrics = item.get("lyrics", "[instrumental]")
    dur = float(item.get("duration", DEFAULT_DURATION))
    steps = int(item.get("steps", DEFAULT_STEPS))
    seed = int(item.get("seed", BASE_SEED + idx if BASE_SEED > 0 else 0))
    try:
        r = run_acestep(
            prompt=prompt,
            lyrics=lyrics,
            audio_duration=dur,
            infer_step=steps,
            seed=seed,
            task="text2music",
            prefix=title,
        )
        generated_files.append(r["wav_path"])
        print(f"  [{idx+1:02d}/{len(items)}] ✓ {title} -> {r['wav_path']} ({r['elapsed_s']:.1f}s)")
    except Exception as e:
        print(f"  [{idx+1:02d}/{len(items)}] ✗ FAILED {title}: {e}")

print(f"\n[Done] {len(generated_files)}/{len(items)} track(s) saved to {OUT_DIR_3D} and {OUT_DIR_TTS}")
if generated_files:
    display(Audio(generated_files[-1], autoplay=False))
    display(FileLink(generated_files[-1]))